In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import monai
from glob import glob
import matplotlib.pyplot as plt
from onekey_algo import get_param_in_cwd

mydir = get_param_in_cwd('data_pattern')
samples = [os.path.join(mydir, f) for f in os.listdir(mydir) if f.endswith('.jpg') or f.endswith('.png')]
samples

In [ ]:
from onekey_algo.custom.components.comp2 import extract, init_from_model, init_from_onekey

model, transformer, device = init_from_onekey(os.path.join(get_param_in_cwd('model_root'), get_param_in_cwd('model_name'), 'viz'))
for n, m in model.named_modules():
    print('Feature name:', n, "|| Module:", m)

In [ ]:
target_layer_candidates = [
    name for name, module in model.named_modules()
    if name.endswith("layer4.1.conv2")
]
if len(target_layer_candidates) != 1:
    raise RuntimeError(
        "Unable to identify a unique Grad-CAM target layer: "
        f"{target_layer_candidates}"
    )
target_layer = target_layer_candidates[0]
gradcam = monai.visualize.GradCAM(nn_module=model, target_layers=target_layer)

In [ ]:
from onekey_algo.datasets.image_loader import default_loader
from onekey_algo.custom.components.comp2 import show_cam_on_image
import torch

for sample in samples:
    img = default_loader(sample)
    sample_ = transformer(img)
    sample_  = sample_.view(1, *sample_.size()).to(device)
    res_cam = gradcam(x=sample_, class_idx=None)
    fig, axes = plt.subplots(1, 2, figsize=(20, 10), facecolor='white')
    axes[0].imshow(res_cam[0][0].cpu(), cmap='jet')
    axes[1].imshow(img.resize(sample_.size()[2:]))
    plt.show()
    
    plt.figure(figsize=(10, 10))
    plt.imshow(show_cam_on_image(img.resize(sample_.size()[2:]), res_cam[0][0].cpu(), use_rgb=True, reverse=False))